# Lab 6.1 · Topics Meet Sentiment
**Chapter 6 · Advanced Topics and Project Kick-off**

**Goal:** find, name and validate topics, then combine them with sentiment — a test run for your project data.
**Data:** `outputs/posts_ch3.parquet` if it exists, otherwise `data/ev_posts.csv`.

📖 Interactive study guide: [Chapter 6](https://mehrdadjalali-ai.github.io/SocialMediaAnayltics/interactive/Chapter6_Advanced_Topics_Project.html) — includes the proposal canvas.

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTD = ROOT / "data", ROOT / "outputs"
OUTD.mkdir(exist_ok=True)
SEED = 42
import re, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
p = OUTD / "posts_ch3.parquet"
df = pd.read_parquet(p) if p.exists() else pd.read_csv(DATA / "ev_posts.csv").drop_duplicates("text")
if "text_classical" not in df:
    df["text_classical"] = df["text"].str.replace(r"https?://\S+", " ", regex=True).str.lower()
print(len(df), "posts")

2363 posts


## Walkthrough · NMF topics

In [2]:
my_stopwords = sorted((set(ENGLISH_STOP_WORDS) - {"not", "no", "never"}) | {"ev", "url", "user"})
tfidf = TfidfVectorizer(min_df=5, max_df=0.5, stop_words=my_stopwords, token_pattern=r"(?u)\b[^\W\d_]{3,}\b")
X = tfidf.fit_transform(df["text_classical"])
nmf = NMF(n_components=6, random_state=SEED, max_iter=500)
W = nmf.fit_transform(X)                       # posts x topics
df["topic"] = W.argmax(axis=1)
words = tfidf.get_feature_names_out()
for k, comp in enumerate(nmf.components_):
    print(k, [words[i] for i in comp.argsort()[-8:][::-1]])

0 ['added', 'useful', 'journey', 'gave', 'explanation', 'time', 'disappointed', 'days']
1 ['simple', 'avoidable', 'problem', 'turned', 'impressed', 'not', 'expected', 'today']
2 ['planning', 'arrives', 'sharing', 'says', 'test', 'dealer', 'monday', 'drive']
3 ['logged', 'recorded', 'compare', 'week', 'today', 'app', 'charging', 'efficiency']
4 ['petrol', 'stressful', 'smooth', 'old', 'practical', 'car', 'better', 'expected']
5 ['die', 'und', 'auto', 'fahrt', 'beim', 'dem', 'heute', 'mit']


In [3]:
pd.crosstab(df["topic"], df["sentiment"], normalize="index").round(2)

sentiment,negative,neutral,positive
topic,,,
0,1.00,0.00,0.00
1,0.91,0.03,0.06
2,0.01,0.68,0.31
3,0.06,0.88,0.06
4,0.00,0.00,1.00
5,0.50,0.22,0.28


## Challenge 1 · Name the topics

Read **5 representative posts** per topic (highest weight in W). Give each topic a name and a one-line definition. Try k = 4 … 10 and keep the k you can name.

> 💡 **Hint:** `df.assign(w=W[:, k]).sort_values('w', ascending=False)['text'].head(5)`

In [4]:
# Challenge 1 — your code here


**Your interpretation (Challenge 1):** _write 2–3 sentences — what did you find, and what could explain it?_

## Challenge 2 · Topics × sentiment

Which topic drives dissatisfaction? Report the share of negative posts per topic **with n**.

> 💡 **Hint:** `pd.crosstab(df.topic, df.sentiment, margins=True)`

In [5]:
# Challenge 2 — your code here


**Your interpretation (Challenge 2):** _write 2–3 sentences — what did you find, and what could explain it?_

## Challenge 3 · Integrated evidence

Support **one finding** with two other perspectives (emotion, network or diffusion — from your earlier labs).

> 💡 **Hint:** E.g. negative topic → emotion distribution (Lab 3) → which community posts it (Lab 4).

In [6]:
# Challenge 3 — your code here


**Your interpretation (Challenge 3):** _write 2–3 sentences — what did you find, and what could explain it?_

## Bring to the project
Use this notebook as a template for your project data — hand it in together with your project proposal canvas.